In [2]:
import numpy as np
from scipy.optimize import approx_fprime
import plotly.graph_objects as go

In [3]:
def rastrigin(x, A=10):
    x = np.asarray(x)
    x0, x1 = x[0], x[1]
    return A*2 + (x0**2 - A*np.cos(2*np.pi*x0)) + (x1**2 - A*np.cos(2*np.pi*x1))

In [4]:
def easom(x):
    x = np.asarray(x)
    x0, x1 = x[0], x[1]
    return (-np.cos(x0) * np.cos(x1)) * np.exp(-((x0 - np.pi)**2 + (x1 - np.pi)**2))

In [5]:
def levy_13(x):
    x = np.asarray(x)
    x0, x1 = x[0], x[1]  
    term1 = np.sin(3 * np.pi * x0) ** 2
    term2 = ((x0 - 1) ** 2) * (1 + np.sin(3 * np.pi * x1) ** 2)
    term3 = ((x1 - 1) ** 2) * (1 + np.sin(2 * np.pi * x1) ** 2)
    return term1 + term2 + term3

In [6]:
def gradient(func):
    return lambda x: approx_fprime(np.asarray(x, dtype=float), func, 1e-6)

In [39]:
def plot3d(f, path, x_star, x_res, bounds=(-2,2,-2,2), title='', num=100, color='viridis'):
    xs, ys = np.linspace(bounds[0], bounds[1], num), np.linspace(bounds[2], bounds[3], num)
    X, Y = np.meshgrid(xs, ys); Z = f([X, Y]); Zp = np.array([f(p) for p in path])
    fig = go.Figure([
        go.Surface(x=X, y=Y, z=Z, opacity=.9, colorscale=color, showscale=False),
        go.Scatter3d(x=path[:,0], y=path[:,1], z=Zp, mode='lines+markers',
                     line=dict(color='blue', width=6), marker=dict(size=3), name='trajectory'),
        go.Scatter3d(x=[x_star[0]], y=[x_star[1]], z=[f(x_star)], mode='markers',
                     marker=dict(size=10, color='red', symbol='diamond'), name='x*'),
        go.Scatter3d(x=[x_res[0]], y=[x_res[1]], z=[f(x_res)], mode='markers', marker=dict(size=10, color='green', symbol='diamond'), name='res'),
    ])
    fig.update_layout(title=title, width=950, height=700,
                      scene=dict(aspectmode='cube', camera=dict(eye=dict(x=1.6,y=-1.6,z=1.0))),
                      margin=dict(l=0,r=0,b=0,t=40))
    fig.show(renderer='iframe')

In [8]:
def dich(func, a0, b0, l=0.001, eps=1e-6):
    # start = time.time()
    a_k = a0
    b_k = b0
    trace = []
    while True:
        y_k = (a_k+b_k-eps)/2
        z_k = (a_k+b_k+eps)/2
        if func(y_k) <= func(z_k):
            a_next = a_k
            b_next = z_k
        else:
            a_next = y_k
            b_next = b_k
        trace.append((a_next + b_next) / 2)
        L_next = abs(b_next-a_next)
        if L_next <= l:
            # end = time.time()
            # print(f'Execution time of dichotomy: {end - start} ms')
            return (a_next + b_next) / 2, trace
        else:
            a_k = a_next
            b_k = b_next

def fix(f, x, t, i):
    temp = x.copy()
    temp[i] = t
    return f(temp)

def descent(f, x0=[0.0, 0.0], limit=1000, eps=1e-5, delta=1.0):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for _ in range(limit):
        x_old = x.copy()
        for i in range(len(x)):
            g = lambda t, i=i: fix(f, x, t, i)
            t_opt, trace = dich(g, x[i] - delta, x[i] + delta,
                                l=eps * 10, eps=eps)
            for t in trace[::5]:
                x_step = x.copy()
                x_step[i] = t
                path.append(x_step)
            x[i] = t_opt
        path.append(x.copy())
        if np.linalg.norm(x - x_old) < eps:
            break
    return x, f(x), np.array(path)

In [48]:
def davidon_and_co(f, grad, x0=[0.0, 0.0], eps=0.0001, limit=1000):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    H = np.eye(len(x))
    for _ in range(limit):
        g = grad(x)
        if np.linalg.norm(g) < eps:
            break
        p = np.dot(-H, g)
        second = lambda a: f(x + a * p)
        alpha, trace = dich(second, 0.0, 1.0)
        for t in trace:
            path.append(x+t*p)
        s = alpha * p
        x_new = x + s
        y = grad(x_new) - g
        Hy = np.dot(H, y)
        H = H + np.outer(s, s) / np.dot(s, y) - np.outer(Hy, Hy) / np.dot(y, Hy)
        x = x_new
        path.append(x.copy())
    return x, f(x), np.array(path)

In [49]:
x_rast, f_rast, path_rast = descent(rastrigin, [-2.0, 1.0])
plot3d(rastrigin, path_rast, [0,0], x_rast, title="Растригин + Спуск", color='turbo')
print(f'{x_rast}')

[-1.e-05  5.e-06]


In [10]:
x_rast, f_rast, path_rast = davidon_and_co(rastrigin, gradient(rastrigin), [-1.0, 1.5])
plot3d(rastrigin, path_rast, [0,0], x_rast, title="Растригин + DFP", bounds=(-10, 10, -10, 10))
print(f'{x_rast}')

[ 2.98485534 -3.97978435]


In [11]:
x_eas, f_eas, path_eas = descent(easom, [-2.0, 1.0])
plot3d(easom, path_eas, [3.14,3.14], x_eas, title="Изом + Спуск", bounds=(-10, 10, -10, 10))
print(f'{x_eas}')

[-4.77543778  1.30499115]


In [12]:
x_eas, f_eas, path_eas = davidon_and_co(easom, gradient(easom), [-2.0, 1.0])
plot3d(easom, path_eas, [3.14,3.14], x_eas, title="Изом + DFP", bounds=(-10, 10, -10, 10))
print(f'{x_eas}')

[-2.  1.]


In [13]:
x_lev, f_lev, path_lev = descent(levy_13, [-2.0, 1.0])
plot3d(levy_13, path_lev, [1,1], x_lev, title="Леви №13 + Спуск", bounds=(-10, 10, -10, 10))
print(f'{x_lev}')

[0.6716175 1.       ]


In [14]:
x_lev, f_lev, path_lev = davidon_and_co(levy_13, gradient(levy_13), [-2.0, 1.0])
plot3d(levy_13, path_lev, [1.0,1.0], x_lev, title="Леви №13 + DFP", bounds=(-10, 10, -10, 10))
print(f'{x_lev}')

[1.32961948 0.99999949]


In [46]:
def nesterov(f, grad, x0=[0.0, 0.0], limit=1000, eps=1e-5, beta=0.5, alpha=0.009):
    x = np.array(x0, dtype=float)
    v = np.zeros_like(x)
    path = [x.copy()]
    
    for k in range(limit):
        x_ahead = x + beta * v
        g = grad(x_ahead)
        v = beta * v - alpha * g
        x_new = x + v
        path.append(x_new.copy())
        if np.linalg.norm(x_new - x) < eps:
            x = x_new
            break
        x = x_new
    return x, f(x), np.array(path)

In [47]:
x_rast, f_rast, path_rast = nesterov(rastrigin, gradient(rastrigin), [-8.0, 1.0], beta=0.4, alpha=0.008)
plot3d(rastrigin, path_rast, [0,0], x_rast, title="Растригин + Ускоренный градиент Нестерова", bounds=(-10, 10, -10, 10))
print(f'{x_rast}')

[-0.19799263 -0.20340767]


In [17]:
x_eas, f_eas, path_eas = nesterov(easom, gradient(easom), [-2.0, 1.0], beta=2, alpha=0.2)
plot3d(easom, path_eas, [3.14,3.14], x_eas, title="Изом + Ускоренный градиент Нестерова", bounds=(-10, 10, -10, 10))
print(f'{x_eas}')

[-2.  1.]


In [18]:
x_lev, f_lev, path_lev = nesterov(levy_13, gradient(levy_13), [-2.0, 1.0], beta=0.8, alpha=0.012)
plot3d(levy_13, path_lev, [1,1], x_lev, title="Леви №13 + Ускоренный градиент Нестерова", bounds=(-10, 10, -10, 10))
print(f'{x_lev}')

[1.29999664 0.9999995 ]


In [59]:
def adagrad(f, grad, x0=[0.0,0.0], limit=5000, alpha=0.01, eps=1e-6, t=1e-5):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    s = np.zeros_like(x)
    for k in range(limit):
        g = grad(x)
        if np.linalg.norm(g) < t:
            print('a')
            break
        s += g * g
        x_new = x - alpha * g / (eps + np.sqrt(s))
        path.append(x_new.copy())
        if np.linalg.norm(x_new - x) < t:
            x = x_new
            print('b')
            break
        x = x_new
    return x, f(x), np.array(path)

In [55]:
x_rast, f_rast, path_rast = adagrad(rastrigin, gradient(rastrigin), [-2.5, 1.3], eps=1e-6, alpha=0.1)
plot3d(rastrigin, path_rast, [0,0], x_rast, title='Растригин + Adagrad', bounds=(-10, 10, -10, 10))
print(f'{x_rast}')

b


[-1.98993164  0.99495884]


In [21]:
x_eas, f_eas, path_eas = adagrad(easom, gradient(easom), [2.0, 1.6])
plot3d(easom, path_eas, [3.14,3.14], x_eas, title="Изом + Adagrad", bounds=(-10, 10, -10, 10))
print(f'{x_eas}')

[3.141591   3.11637151]


In [22]:
x_lev, f_lev, path_lev = adagrad(levy_13, gradient(levy_13), [-2.0, 1.0], limit=5000, alpha=0.5)
plot3d(levy_13, path_lev, [1,1], x_lev, title="Леви №13 + Adagrad", bounds=(-10, 10, -10, 10))
print(f'{x_lev}')

[0.0112234  0.67006735]


In [24]:
def adam(f, grad, x0=[0.0, 0.0], limit=5000, t=1e-5, alpha=0.001, gamma1=0.9, gamma2=0.999, eps=1e-8):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    v = np.zeros_like(x)
    s = np.zeros_like(x)
    for k in range(1, limit+1):
        g = grad(x)
        if np.linalg.norm(g) < t:
            break
        v = gamma1 * v + (1-gamma1) * g
        s = gamma2 * s + (1-gamma2) * (g * g)
        v_cor = v / (1 - gamma1 ** k)
        s_cor = s / (1 - gamma2 ** k)
        x_new = x - alpha * v_cor / (np.sqrt(s_cor) + eps)
        path.append(x_new.copy())
        if np.linalg.norm(x_new - x) < t:
            x = x_new
            break   
        x = x_new
    return x, f(x), np.array(path)

In [41]:
x_rast, f_rast, path_rast = adam(rastrigin, gradient(rastrigin), [-2.5, 8.0], alpha=0.9)
plot3d(rastrigin, path_rast, [0,0], x_rast, title='Растригин + Adam', bounds=(-10, 10, -10, 10), color='inferno')
print(f'{x_rast}')

[-1.95984998 -0.01096507]


In [42]:
x_eas, f_eas, path_eas = adam(easom, gradient(easom), [-2.0, 1.0], alpha=0.9)
plot3d(easom, path_eas, [3.14,3.14], x_eas, title="Изом + Adam", bounds=(-10, 10, -10, 10))
print(f'{x_eas}')

[-2.  1.]


In [44]:
x_lev, f_lev, path_lev = adam(levy_13, gradient(levy_13), [-2.0, 1.0], limit=5000, alpha=0.5)
plot3d(levy_13, path_lev, [1,1], x_lev, title="Леви №13 + Adam", bounds=(-10, 10, -10, 10))
print(f'{x_lev}')

[1.32962183 0.99996347]
